# 第 23 节: 结课项目

## 目标
综合运用所学知识, 完成一个完整的 RL 项目

## 项目要求
1. 选择一个 Gymnasium 环境 (推荐 CartPole, Acrobot, MountainCar)
2. 定义或修改 observation, action, reward
3. 实现或复用 PPO/A2C/DQN
4. 训练至少 3 个随机种子
5. 绘制训练曲线 (均值 +/- 标准差)
6. 保存最优模型
7. 生成智能体运行视频
8. 完成至少 2 项消融实验
9. 分析至少 1 个失败案例
10. 撰写简短实验报告 (Markdown 在 Notebook 中)

## 项目模板

In [ ]:
import sys; sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath("__file__")))), os
from rl_course.utils.seeding import set_seed
from rl_course.agents.ppo import PPOAgent
import gymnasium as gym; import numpy as np
import torch
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ===== 步骤 1: 选择环境 =====
ENV_NAME = "CartPole-v1"
N_SEEDS = 3

# ===== 步骤 2: 训练函数框架 =====
def train_ppo(env_name, seed, total_steps=100_000):
    set_seed(seed)
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n

    agent = PPOAgent(
        state_dim=state_dim, n_actions=n_actions,
        hidden_dims=[64, 64], n_steps=512,
        batch_size=64, n_epochs=10, lr=3e-4,
        gamma=0.99, gae_lambda=0.95, clip_epsilon=0.2,
    )

    all_returns = []
    state, _ = env.reset()
    ep_return = 0.0

    for step in range(total_steps):
        action = agent.act(state, train=True)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # 计算 next_value（必须在 env.reset() 之前！）
        with torch.no_grad():
            if terminated:
                next_value = 0.0
            else:
                next_state_t = torch.as_tensor(
                    next_state, dtype=torch.float32, device=agent.device
                ).unsqueeze(0)
                next_value = agent.network.get_value(next_state_t).item()

        agent.store(reward, done, terminated, next_value)
        ep_return += reward
        state = next_state

        if done:
            state, _ = env.reset()
            all_returns.append(ep_return)
            ep_return = 0.0

        # PPO update when buffer is full
        if agent.buffer.full:
            agent.update()

        if (step + 1) % 20000 == 0:
            avg = np.mean(all_returns[-10:]) if all_returns else 0
            print(f"  Step {step+1:6d} | Avg Return: {avg:.1f}")

    env.close()
    return agent, all_returns

print("项目模板已加载 - 根据你的环境修改 train_ppo 函数")
print("然后运行多种子训练和消融实验")


## 实验报告模板

### 1. 环境描述
### 2. 算法选择与超参数
### 3. 训练结果
### 4. 消融实验
### 5. 失败案例分析
### 6. 结论

## 评分标准

| 项目 | 分值 |
|------|------|
| 训练结果达到合理水平 | 30 |
| 多种子均值+标准差报告 | 10 |
| 至少 2 项消融实验 | 20 |
| 失败案例分析 | 15 |
| 视频/动画演示 | 10 |
| 实验报告质量 | 15 |

---
下一节: 24_interview_review.ipynb